# 06 — Exposure Normalisation

## Why this notebook exists

Every finding so far has been conditional on a crash having occurred. That was
deliberate: crash counts are not risk, because
crashes(s) = per-passage risk(s) × passages(s)
and only the product is observed. Mitte records the most bicycle crashes in
Berlin because Mitte has the most cyclists.

This notebook introduces the missing denominator.

## What the counter data can and cannot do

Berlin operates **35 permanent bicycle counters** (*Dauerzählstellen*) with
hourly readings since 2012, published by SenUMVK with GPS coordinates for each
station.Of the 35, 24 have usable hourly series across the window — three IDs are test
periods or renamed duplicates (§4.4) and the rest lack coverage — and 22 survive
the coordinate-error exclusion applied in §5.

35 stations against 238,972 network segments means per-segment exposure is not
measurable. Two things are:

1. **A temporal profile** — hour, weekday and month multipliers. Solid from 35
   stations, because the temporal pattern of cycling is broadly similar across
   the city.
2. **35 spatial anchor points** — enough to calibrate a class-level multiplier
   (`cycleway` high, `service` near zero), not enough for per-segment estimates.

## Why this matters for the model

Time features currently score AUC ≈ 0.500 in the occurrence model, because
`build_ml_dataset` draws negative sample times from the positive time pool. An
exposure-normalised crash rate per time band is a legitimate feature that does
not depend on that sampling design.

## Known biases

**Station siting is not random.** Jannowitzbrücke, Oberbaumbrücke, Frankfurter
Allee — the counters sit on major cycling corridors, not residential streets.
The resulting profile is commuter-weighted, so morning and evening peaks are
likely sharper than the city-wide truth.

**Zero is not missing.** Per the `Legende` sheet, `0` means a measurement was
taken and counted zero; a blank means no data. Conflating them would make outage
periods look like low volume.

In [1]:
import re

import numpy as np
import pandas as pd

XL = "../data/raw/gesamtdatei-stundenwerte.xlsx"
YEARS = range(2018, 2026)  # match the Berlin accident window


def load_counter_year(year: int) -> pd.DataFrame:
    """One year of hourly counts in long format.

    Headers carry the installation date, separated by a newline in some year
    sheets and by a space in others ("02-MI-JAN-N 01.04.2015"). Keep only the
    leading station ID, or the same counter appears as two stations.
    """
    d = pd.read_excel(XL, sheet_name=f"Jahresdatei {year}")
    d = d.rename(columns={d.columns[0]: "ts"})
    d.columns = ["ts"] + [re.split(r"[\s\n\r]", str(c).strip())[0] for c in d.columns[1:]]
    d["ts"] = pd.to_datetime(d["ts"], errors="coerce")
    d = d.dropna(subset=["ts"])

    long = d.melt(id_vars="ts", var_name="station", value_name="count")
    long["count"] = pd.to_numeric(long["count"], errors="coerce")
    return long.dropna(subset=["count"])


counts = pd.concat([load_counter_year(y) for y in YEARS], ignore_index=True)
counts["hour"] = counts["ts"].dt.hour
counts["month"] = counts["ts"].dt.month
counts["dow_iso"] = counts["ts"].dt.dayofweek  # Monday = 0

print(f"{len(counts):,} station-hours | {counts['station'].nunique()} stations")
print(f"range: {counts['ts'].min().date()} to {counts['ts'].max().date()}")
print("\nstation-hours per year:")
print(counts.groupby(counts["ts"].dt.year).size().to_string())

1,991,409 station-hours | 38 stations
range: 2018-01-01 to 2025-12-31

station-hours per year:
ts
2018    227738
2019    227702
2020    226415
2021    218169
2022    221182
2023    266471
2024    301558
2025    302174


---
## 1. Temporal profile

Direction suffixes are collapsed first: total passages at Jannowitzbrücke is
north plus south, not their mean. This matters for absolute volume, though not
for the shape of the temporal profile.

Stations with fewer than 2,000 recorded hours are excluded — these are test
periods and renamed IDs. Berlin's own data carries two such inconsistencies
(Karl-Marx-Allee appears as both `01-MI-AL-W` and `02-MI-AL-W`, Nonnendammallee
as both `03-SP-NO` and `16-SP-NO`).

The profile is computed as the **mean count per station-hour**, not a sum across
stations. Summing would let the number of active counters drive the level:
station-hours rise from 227,738 in 2018 to 302,174 in 2025 as new counters came
online.

In [2]:
# Drop test periods and renamed IDs before profiling.
MIN_HOURS = 2000
hours_per_station = counts.groupby("station").size()
keep = hours_per_station[hours_per_station >= MIN_HOURS].index
dropped = hours_per_station[hours_per_station < MIN_HOURS]

if len(dropped):
    print(f"Excluded {len(dropped)} short-lived station IDs:")
    print(dropped.to_string(), "\n")

c = counts[counts["station"].isin(keep)].copy()

# Collapse direction suffixes: -N/-S/-O/-W mark the two sides of one location.
c["location"] = c["station"].str.replace(r"-[NSOW]$", "", regex=True)
print(f"{c['station'].nunique()} stations across {c['location'].nunique()} locations")

# Mean per station-hour, so the growing number of counters does not drive the
# level. Only the relative shape is used downstream.
hour_profile = c.groupby("hour")["count"].mean()
hour_index = hour_profile / hour_profile.mean()

print("\nHour of day: mean count per station-hour, and index vs daily mean")
for h in range(24):
    print(f"  {h:02d}  {hour_profile[h]:7.1f}   {hour_index[h]:.3f}")

print(f"\npeak {hour_profile.idxmax():02d}:00 = {hour_profile.max():.0f}")
print(f"trough {hour_profile.idxmin():02d}:00 = {hour_profile.min():.0f}")
print(f"ratio = {hour_profile.max() / hour_profile.min():.1f}x")

Excluded 3 short-lived station IDs:
station
02-MI-AL-W    368
03-SP-NO-O    948
03-SP-NO-W    948 

35 stations across 24 locations

Hour of day: mean count per station-hour, and index vs daily mean
  00     22.5   0.246
  01     12.6   0.138
  02      7.5   0.082
  03      5.3   0.058
  04      6.5   0.072
  05     17.5   0.192
  06     37.4   0.409
  07    100.9   1.103
  08    161.7   1.769
  09    135.0   1.477
  10     97.2   1.063
  11     99.9   1.093
  12    113.3   1.240
  13    125.8   1.376
  14    140.0   1.532
  15    163.3   1.786
  16    171.6   1.877
  17    185.2   2.026
  18    188.3   2.060
  19    141.2   1.545
  20     95.8   1.048
  21     70.6   0.772
  22     56.6   0.619
  23     38.4   0.420

peak 18:00 = 188
trough 03:00 = 5
ratio = 35.7x


---
## 2. Exposure-normalised crash frequency

Crash counts alone measure where and when cyclists are. Dividing by counter
volume recovers the first factor of
crashes(t) = per-passage risk(t) × passages(t)

The index is relative to the citywide mean, so a value above 1.0 means more
crashes **per passage** than average — not more crashes.

In [4]:
DTYPES = {"year": "int16", "month": "int8", "hour": "int8",
          "accident_severity": "int8", "is_ksi": "int8"}

acc = pd.read_csv("../data/processed/berlin_bike_2018_2025.csv")
acc["is_ksi"] = acc["accident_severity"].isin([1, 2]).astype(int)
print(f"{len(acc):,} crashes | KSI {acc['is_ksi'].mean():.4f}")

# Crashes and exposure aggregated on the same hour-of-day grid.
crash_by_hour = acc.groupby("hour").size()
expo_by_hour = c.groupby("hour")["count"].mean()

comp = pd.DataFrame({
    "crashes": crash_by_hour,
    "exposure": expo_by_hour,
}).dropna()

# Rate per unit exposure, indexed to the citywide mean so the arbitrary units
# of the counter data cancel out.
comp["rate"] = comp["crashes"] / comp["exposure"]
comp["freq_index"] = comp["rate"] / (comp["crashes"].sum() / comp["exposure"].sum())

ksi_by_hour = acc.groupby("hour")["is_ksi"].mean()
comp["ksi_rate"] = ksi_by_hour
comp["expected_harm"] = comp["freq_index"] * comp["ksi_rate"]

print("\n hour  crashes  exposure  freq_index  ksi_rate  expected_harm")
for h in comp.index:
    r = comp.loc[h]
    print(f"  {h:02d}   {r['crashes']:6.0f}  {r['exposure']:8.1f}     "
          f"{r['freq_index']:6.3f}    {r['ksi_rate']:6.4f}    {r['expected_harm']:6.4f}")

print(f"\nfreq_index:     max {comp['freq_index'].max():.2f} at "
      f"{comp['freq_index'].idxmax():02d}:00, min {comp['freq_index'].min():.2f} at "
      f"{comp['freq_index'].idxmin():02d}:00")
print(f"expected_harm:  max {comp['expected_harm'].max():.4f} at "
      f"{comp['expected_harm'].idxmax():02d}:00, min {comp['expected_harm'].min():.4f} at "
      f"{comp['expected_harm'].idxmin():02d}:00")
print(f"expected_harm spread: {comp['expected_harm'].max() / comp['expected_harm'].min():.2f}x")

37,948 crashes | KSI 0.1299

 hour  crashes  exposure  freq_index  ksi_rate  expected_harm
  00      269      22.5      0.692    0.2156    0.1491
  01      188      12.6      0.864    0.2287    0.1977
  02      112       7.5      0.864    0.2054    0.1773
  03       87       5.3      0.952    0.2184    0.2080
  04       80       6.5      0.707    0.2625    0.1856
  05      302      17.5      0.997    0.1556    0.1551
  06      778      37.4      1.204    0.1375    0.1656
  07     2124     100.9      1.217    0.1191    0.1450
  08     2734     161.7      0.978    0.1145    0.1119
  09     2363     135.0      1.012    0.1176    0.1191
  10     1893      97.2      1.126    0.1315    0.1482
  11     1895      99.9      1.097    0.1361    0.1493
  12     2187     113.3      1.116    0.1358    0.1515
  13     2448     125.8      1.125    0.1291    0.1452
  14     2754     140.0      1.137    0.1260    0.1433
  15     3387     163.3      1.199    0.1231    0.1477
  16     3341     171.6      

In [5]:

import math


def wilson_ci(successes, n, z=1.96):
    """Wilson score interval for a proportion. Behaves sensibly for small
    counts, unlike the normal approximation."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = successes / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (centre - margin, centre + margin)

BANDS = {
    "night 22-05":   [22, 23, 0, 1, 2, 3, 4, 5],
    "morning 06-09": [6, 7, 8, 9],
    "midday 10-15":  [10, 11, 12, 13, 14, 15],
    "evening 16-21": [16, 17, 18, 19, 20, 21],
}

rows = []
overall = comp["crashes"].sum() / comp["exposure"].sum()
for label, hrs in BANDS.items():
    sub = comp.loc[comp.index.isin(hrs)]
    cr, ex = sub["crashes"].sum(), sub["exposure"].sum()
    ksi = acc.loc[acc["hour"].isin(hrs), "is_ksi"]
    lo, hi = wilson_ci(int(ksi.sum()), len(ksi))
    rows.append({
        "band": label, "crashes": int(cr), "exposure": round(ex, 1),
        "freq_index": round((cr / ex) / overall, 3),
        "ksi_rate": round(ksi.mean(), 4),
        "ksi_lo": round(lo, 4), "ksi_hi": round(hi, 4),
        "expected_harm": round((cr / ex) / overall * ksi.mean(), 4),
    })

band = pd.DataFrame(rows)
print(band.to_string(index=False))
print(f"\nexpected_harm spread across bands: "
      f"{band['expected_harm'].max() / band['expected_harm'].min():.2f}x")

         band  crashes  exposure  freq_index  ksi_rate  ksi_lo  ksi_hi  expected_harm
  night 22-05     2083     166.9       0.722    0.1839  0.1678  0.2011         0.1327
morning 06-09     7999     434.9       1.063    0.1189  0.1120  0.1262         0.1264
 midday 10-15    14564     739.5       1.139    0.1294  0.1240  0.1349         0.1473
evening 16-21    13302     852.8       0.902    0.1287  0.1231  0.1345         0.1161

expected_harm spread across bands: 1.27x


---
## 3. Findings

### 3.1 Crash frequency per passage is nearly flat across the day

Exposure varies by **35.7×** between the 03:00 trough and the 18:00 peak. Crash
counts vary by roughly 7× across the same span. But crashes *per passage* vary by
only 2× at hourly resolution, and by 1.6× when grouped into bands:

| Band | Crashes | Exposure | Frequency index | KSI rate | Expected harm |
|---|---|---|---|---|---|
| night 22–05 | 2,083 | 166.9 | **0.722** | **18.39%** [16.78–20.11] | 0.1327 |
| morning 06–09 | 7,999 | 434.9 | 1.063 | 11.89% [11.20–12.62] | 0.1264 |
| midday 10–15 | 14,564 | 739.5 | **1.139** | 12.94% [12.40–13.49] | **0.1473** |
| evening 16–21 | 13,302 | 852.8 | 0.902 | 12.87% [12.31–13.45] | 0.1161 |

The variation in raw crash counts is almost entirely volume. This is the clearest
demonstration in the project that counts are not risk.

### 3.2 Night is safer per passage and more severe per crash

Cyclists have **28% fewer crashes per passage** at night than the daily average,
but those crashes carry a **55% higher KSI rate** than the morning. The intervals
do not overlap.

Neither fact is visible in crash counts alone: night accounts for 2,083 crashes
against 14,564 at midday, which reads as "night is safe" until the denominator is
introduced.

### 3.3 Expected harm per passage varies by only 1.27× across the day

Multiplying frequency by severity, the two effects largely cancel: midday is
highest at 0.1473, evening lowest at 0.1161.

**This closes the question of the hour input in the routing engine.** A
time-of-day multiplier of at most 1.27× applies equally to every segment, so it
cannot change the ordering of edge costs — and Dijkstra compares only the
ordering. The hour slider not changing the route is correct behaviour, not a
missing feature.

Three independent lines of evidence now agree: a time-only occurrence model
scores lift 1.00 (`03_model_leakage_fix`), the risk ranking of road classes is
invariant across time windows at Spearman 1.000 (`04_severity_model`), and
exposure-normalised expected harm varies by 1.27×.

### 3.4 What the night finding is good for

Not routing between segments, but a hard constraint. Excluding unlit `path` and
`service` segments during 22:00–05:00 is now supported by measurement: those
hours carry the highest severity, and the frequency model scores those classes at
near zero purely because nobody rides them.

---
## 4. Limitations

1. **Station siting is not random.** Jannowitzbrücke, Oberbaumbrücke, Frankfurter
   Allee — the 35 counters sit on major cycling corridors, not residential
   streets. The temporal profile is commuter-weighted, so morning and evening
   peaks are likely sharper than the city-wide truth, and the night trough
   possibly deeper.
2. **Counters measure 24 locations, crashes cover 238,972 segments.** The
   normalisation is temporal only. Spatial exposure remains unmeasured, so the
   `service` / `path` problem is not solved by this notebook.
3. **Counter units are arbitrary.** The frequency index is relative to the
   citywide mean; absolute crash rates per cyclist-kilometre are not computed,
   since segment length at the counter locations is not accounted for.
4. **Three station IDs were excluded** as test periods or renamed duplicates
   (`02-MI-AL-W`, `03-SP-NO-O`, `03-SP-NO-W`). Berlin's own data carries district
   prefix inconsistencies for Karl-Marx-Allee and Nonnendammallee.
5. **Night severity is confounded.** Darkness, speed, alcohol and a different
   rider population coincide, and alcohol and speed are absent from the data. The
   18.39% night KSI rate cannot be attributed to darkness alone.
   The 18.39% uses the 22–05 band of this notebook's partition;
   `04_severity_model` reports 18.79% for `is_night` (22–04), the pipeline's
   definition. Same effect, one hour apart.
6. **Counter coverage grows over the window** — 227,738 station-hours in 2018
   against 302,174 in 2025. The profile uses the mean per station-hour rather
   than a sum to avoid this driving the level, but the station mix still changes.

---
## 5. Spatial exposure: a class-level multiplier

The temporal normalisation above says nothing about which street is dangerous. A
busy protected lane with 10 crashes and a deserted service road with 8 crashes
have very different per-passage risk, and crash counts cannot tell them apart.

24 counter locations against 238,972 segments cannot give per-segment exposure.
What they can give is a **class-level multiplier**: if `cycleway` locations carry
on average N times the volume of `residential` locations, that ratio can be
applied network-wide.

The limits of this are severe and stated up front. The counters sit on major
cycling corridors, so `service`, `path` and `track` have no observations at all —
their multipliers are extrapolated, not measured.

In [6]:
# Station coordinates. IDs here carry no date suffix, so they join directly.
loc = pd.read_excel(XL, sheet_name="Standortdaten")
loc.columns = ["station", "description", "lat", "lon", "installed"]
loc["station"] = loc["station"].astype(str).str.strip()
loc = loc.dropna(subset=["lat", "lon"])
loc["location"] = loc["station"].str.replace(r"-[NSOW]$", "", regex=True)

print(f"{len(loc)} station records, {loc['location'].nunique()} locations")

# Mean hourly passages per location: sum across its direction counters, since a
# bridge carries north plus south, then average over hours.
per_station = c.groupby("station")["count"].mean()
station_to_loc = dict(zip(loc["station"], loc["location"]))

vol = (
    per_station.rename("mean_hourly")
    .to_frame()
    .assign(location=lambda d: d.index.map(station_to_loc))
    .dropna(subset=["location"])
    .groupby("location")["mean_hourly"]
    .sum()
)

# One coordinate per location: the mean of its direction counters.
coords = loc.groupby("location")[["lat", "lon"]].mean()
sites = coords.join(vol.rename("mean_hourly"), how="inner").reset_index()

print(f"\n{len(sites)} locations with both coordinates and volume\n")
print(sites.sort_values("mean_hourly", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

35 station records, 24 locations

23 locations with both coordinates and volume

 location     lat     lon  mean_hourly
05-FK-OBB 52.5012 13.4450     338.1482
02-MI-JAN 52.5139 13.4177     305.3659
10-PA-BER 52.5669 13.4123     236.6303
18-TS-YOR 52.4921 13.3733     211.7095
21-NK-MAY 52.4930 13.4297     198.2334
 14-CW-JU 52.5129 13.3267     178.5560
06-FK-FRA 52.5137 13.4743     178.3554
03-MI-SAN 52.5274 13.3726     170.6415
19-TS-MON 52.4881 13.3698     148.9838
 09-PA-SA 52.5430 13.4121     140.4238
 07-FK-ST 52.5189 13.4261     135.9262
 11-PA-SE 52.5314 13.4124     132.2782
 01-MI-AL 52.5219 13.4175     120.8846
26-LI-PUP 52.5003 13.4743     119.4734
12-PA-SCH 52.5491 13.4004      94.4046
13-CW-PRI 52.4881 13.3331      63.4801
15-SP-KLO 52.5334 13.1988      61.0107
23-TK-KAI 52.4573 13.5187      58.8581
27-RE-MAR 52.5582 13.3649      58.6821
20-TS-MAR 52.4383 13.3879      52.3838
 16-SP-NO 52.5382 13.2475      38.6376
 04-MI-NO 52.5139 13.4179      36.4405
24-MH-ALB 52.4925 13.5

In [7]:
# Berlin's own Standortdaten sheet gives 04-MI-NO (Nordufer) the same
# coordinates as 02-MI-JAN (Jannowitzbrücke), 52.5139 / 13.4179. Nordufer is in
# Wedding, several kilometres away, so this is a copy-paste error in the source
# file. Excluded rather than snapped to the wrong segment.
BAD_COORDS = {"04-MI-NO"}
sites = sites[~sites["location"].isin(BAD_COORDS)].copy()
print(f"{len(sites)} locations after excluding known coordinate errors")

22 locations after excluding known coordinate errors


In [8]:
import geopandas as gpd
import osmnx as ox

# Berlin's own Standortdaten sheet gives 04-MI-NO (Nordufer) the same coordinates
# as 02-MI-JAN (Jannowitzbrücke), 52.5139 / 13.4179. Nordufer is in Wedding,
# several kilometres away, so this is a copy-paste error in the source file.
# Excluded rather than snapped to the wrong segment.
BAD_COORDS = {"04-MI-NO"}
sites = sites[~sites["location"].isin(BAD_COORDS)].copy()
print(f"{len(sites)} locations after excluding known coordinate errors")

Gp = ox.load_graphml("../data/processed/berlin_bike_network_projected.graphml")
edges = pd.read_csv("../data/processed/berlin_osm_edge_features.csv")


pts = gpd.GeoSeries(
    gpd.points_from_xy(sites["lon"], sites["lat"]), crs="EPSG:4326"
).to_crs(Gp.graph["crs"])

# Return format differs across osmnx versions: some return three arrays, others
# a list of (u, v, key) tuples. Unpack the list form.
edge_ids, dist = ox.nearest_edges(
    Gp, X=pts.x.to_numpy(), Y=pts.y.to_numpy(), return_dist=True
)

sites["edge_uid"] = [f"{a}|{b}|{c}" for a, b, c in edge_ids]
sites["snap_dist_m"] = dist

sites = sites.merge(edges[["edge_uid", "highway_simple"]], on="edge_uid", how="left")

print(f"\nsnap distance: median {sites['snap_dist_m'].median():.1f} m, "
      f"max {sites['snap_dist_m'].max():.1f} m\n")
print(sites[["location", "mean_hourly", "highway_simple", "snap_dist_m"]]
      .sort_values("mean_hourly", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:.1f}"))

print("\nMean hourly volume by highway class:")
print(sites.groupby("highway_simple")["mean_hourly"]
      .agg(["size", "mean", "median"]).round(1).to_string())

22 locations after excluding known coordinate errors

snap distance: median 3.0 m, max 16.1 m

 location  mean_hourly highway_simple  snap_dist_m
05-FK-OBB        338.1        primary          3.1
02-MI-JAN        305.4      secondary          4.3
10-PA-BER        236.6        service          0.4
18-TS-YOR        211.7       cycleway          2.5
21-NK-MAY        198.2    residential          1.1
 14-CW-JU        178.6        service         15.1
06-FK-FRA        178.4        primary          1.7
03-MI-SAN        170.6      secondary          5.9
19-TS-MON        149.0    residential          0.3
 09-PA-SA        140.4       cycleway          0.2
 07-FK-ST        135.9        primary          4.9
 11-PA-SE        132.3       cycleway          0.3
 01-MI-AL        120.9        primary         12.2
26-LI-PUP        119.5    residential         16.1
12-PA-SCH         94.4        service          9.1
13-CW-PRI         63.5    residential          0.2
15-SP-KLO         61.0      secondary 

### 5.1 Class-level exposure multipliers cannot be calibrated from these counters

22 locations snapped to the network, median offset 3.0 m, max 16.1 m — geometry
is not the problem.
| Class | Locations | Mean hourly passages |
|---|---|---|
| `secondary` | 3 | 179.0 |
| `service` | 3 | **169.9** |
| `primary` | 5 | 165.1 |
| `cycleway` | 4 | 130.8 |
| `residential` | 6 | 101.7 |
| `path` | 1 | 58.9 |

**The ordering is not usable.** `service` sits third, above `primary` by 4.8
passages per hour on three observations against five — a gap well inside the
noise of samples this size. Berliner Straße, the third busiest location at 236.6
passages per hour, snaps to a `service` segment at 0.4 m.

The counters sit on the separated cycle track beside a main road, and OSM tags that
short parallel way as `service` or `cycleway` rather than inheriting the class of
the road it follows. The class label describes OSM's segmentation of a small piece
of infrastructure, not the traffic environment.

Two further problems make calibration impossible regardless:

**Sample size** — one to six observations per class, `path` has one.

**Selection bias is decisive** — all 22 counters sit on major cycling corridors.
The six locations tagged `residential` (Maybachufer, Monumentenstraße,
Paul-und-Paula-Uferweg) are among Berlin's busiest residential streets, not
typical ones.

**Applying these multipliers would make the routing engine worse.** Assigning
`service` a high volume would raise the denominator on 93,955 service stubs that
carry almost no cyclists, driving their already near-zero risk score lower still.

**Conclusion.** Spatial exposure cannot be estimated from the permanent counter
network. The temporal normalisation in sections 1–3 stands. The spatial dimension
remains unmeasured, and the `service` / `path` problem must be handled with a hard
constraint rather than a learned multiplier.

---
## 6. What can substitute for spatial exposure

Since per-segment exposure is not measurable, the routing cost has to lean on
features that reduce risk independently of volume: protected infrastructure,
fewer junction conflicts, lower speed environment.

All of these are already in the OSM graph. This section checks which tags are
populated well enough to use.

In [9]:
e = ox.graph_to_gdfs(Gp, nodes=False)

TAGS = ["lit", "surface", "smoothness", "cycleway", "cycleway:right",
        "cycleway:left", "oneway:bicycle", "maxspeed", "hgv", "maxweight",
        "railway", "junction", "bicycle", "segregated", "width"]

print(f"{len(e):,} edges\n")
for t in TAGS:
    if t in e.columns:
        cov = e[t].notna().mean()
        print(f"{t:18s} {cov:6.1%}")
        if cov > 0.02:
            top = e[t].astype(str).value_counts().head(4)
            print("  " + top.to_string().replace("\n", "\n  "))
    else:
        print(f"{t:18s} absent")
    print()

441,515 edges

lit                absent

surface            absent

smoothness         absent

cycleway           absent

cycleway:right     absent

cycleway:left      absent

oneway:bicycle     absent

maxspeed            45.9%
  maxspeed
  30    143909
  50     47398
  10      5199
  20      3003

hgv                absent

maxweight          absent

railway            absent

junction             0.1%

bicycle            absent

segregated         absent

width               18.7%
  width
  6      7387
  1      4418
  5      4100
  1.5    4013



In [10]:
import osmnx as ox

# osmnx keeps only a default set of way tags and discards the rest. The
# infrastructure features that matter for routing — lighting, surface, cycleway
# presence, contraflow permission, HGV designation — are not in that default set,
# so the graph has to be re-downloaded with them requested explicitly.
ox.settings.useful_tags_way = [
    "bridge", "tunnel", "oneway", "lanes", "ref", "name", "highway",
    "maxspeed", "service", "access", "area", "junction", "width",
    # added for this project
    "lit", "surface", "smoothness",
    "cycleway", "cycleway:right", "cycleway:left", "cycleway:both",
    "bicycle", "segregated", "oneway:bicycle",
    "hgv", "maxweight", "railway", "tracktype",
]
ox.settings.use_cache = True

G = ox.graph_from_place("Berlin, Germany", network_type="bike", simplify=True)
G_proj = ox.project_graph(G, to_crs="EPSG:25833")

ox.save_graphml(G, "../data/processed/berlin_bike_network_v2.graphml")
ox.save_graphml(G_proj, "../data/processed/berlin_bike_network_projected_v2.graphml")

In [11]:
e2 = ox.graph_to_gdfs(G_proj, nodes=False)
for t in ["lit", "surface", "cycleway", "cycleway:right", "oneway:bicycle",
          "hgv", "railway", "segregated", "bicycle"]:
    cov = e2[t].notna().mean() if t in e2.columns else 0.0
    print(f"{t:18s} {cov:6.1%}")

lit                 63.9%
surface             74.5%
cycleway             3.2%
cycleway:right       5.1%
oneway:bicycle       1.2%
hgv                  1.1%
railway              0.0%
segregated           1.7%
bicycle              6.7%


In [12]:
for t in ["lit", "surface"]:
    print(f"\n{t} — {e2[t].notna().mean():.1%} coverage")
    print(e2[t].astype(str).value_counts().head(10).to_string())


lit — 63.9% coverage
lit
yes                           259247
no                             21979
['no', 'yes']                    893
automatic                         36
['automatic', 'yes']              24
['24/7', 'yes']                   19
24/7                              13
['no', 'automatic', 'yes']        13
['yes', 'no', 'automatic']        12
['no', 'automatic']                7

surface — 74.5% coverage
surface
asphalt                         165470
sett                             38793
paving_stones                    37989
concrete                         28247
dirt                             11682
ground                           11056
['asphalt', 'paving_stones']      5392
['sett', 'asphalt']               4438
compacted                         4242
fine_gravel                       2871


In [13]:
import osmnx as ox

ox.settings.useful_tags_way = [
    "bridge", "tunnel", "oneway", "lanes", "ref", "name", "highway",
    "maxspeed", "service", "access", "area", "junction", "width",
    "lit", "surface", "smoothness",
    "cycleway", "cycleway:right", "cycleway:left", "cycleway:both",
    "bicycle", "segregated", "oneway:bicycle",
    "hgv", "maxweight", "railway", "tracktype",
]
ox.settings.use_cache = True

G2 = ox.graph_from_place("Berlin, Germany", network_type="bike", simplify=True)
G2p = ox.project_graph(G2, to_crs="EPSG:25833")

print(f"{G2.number_of_nodes():,} nodes | {G2.number_of_edges():,} edges")

195,731 nodes | 441,785 edges


In [14]:
ox.save_graphml(G2, "../data/processed/berlin_bike_network_v2.graphml")
ox.save_graphml(G2p, "../data/processed/berlin_bike_network_projected_v2.graphml")
print("saved")

saved


In [15]:
import geopandas as gpd

acc = pd.read_csv("../data/processed/berlin_bike_2018_2025.csv")
acc["is_ksi"] = acc["accident_severity"].isin([1, 2]).astype(int)

pts = gpd.GeoSeries(
    gpd.points_from_xy(acc["longitude"], acc["latitude"]), crs="EPSG:4326"
).to_crs(G2p.graph["crs"])

edge_ids, dist = ox.nearest_edges(
    G2p, X=pts.x.to_numpy(), Y=pts.y.to_numpy(), return_dist=True
)

acc["edge_uid"] = [f"{a}|{b}|{c}" for a, b, c in edge_ids]
acc["snap_dist_m"] = dist

print(f"{len(acc):,} crashes snapped")
print(f"median offset {acc['snap_dist_m'].median():.1f} m, "
      f"within 25 m: {(acc['snap_dist_m'] <= 25).mean():.1%}")

37,948 crashes snapped
median offset 0.7 m, within 25 m: 99.9%


In [16]:
e2r = e2.reset_index()
e2r["edge_uid"] = [f"{u}|{v}|{k}" for u, v, k in zip(e2r["u"], e2r["v"], e2r["key"])]

acc = acc.merge(
    e2r[["edge_uid", "lit", "surface", "highway"]], on="edge_uid", how="left"
)
acc = acc[acc["snap_dist_m"] <= 25].copy()

print(f"{len(acc):,} crashes")
print(f"lit known:     {acc['lit'].notna().mean():.1%}")
print(f"surface known: {acc['surface'].notna().mean():.1%}")

37,896 crashes
lit known:     98.8%
surface known: 99.5%


In [17]:
SURFACE_GROUPS = {
    "smooth": {"asphalt", "concrete", "concrete:plates", "paved"},
    "rough_paved": {"sett", "cobblestone", "paving_stones", "unhewn_cobblestone"},
    "unpaved": {"dirt", "ground", "compacted", "fine_gravel", "gravel", "sand", "grass"},
}


def surface_group(v):
    """Group OSM surface values. simplify=True merges ways, so some edges carry a
    list of conflicting values — treated as unknown rather than forced."""
    s = str(v).lower()
    if s in ("nan", "none") or s.startswith("["):
        return "unknown"
    for name, members in SURFACE_GROUPS.items():
        if s in members:
            return name
    return "other"


def lit_flag(v):
    s = str(v).lower()
    if s in ("nan", "none") or s.startswith("["):
        return "unknown"
    return "lit" if s in ("yes", "24/7", "automatic") else "unlit"


acc["surface_cat"] = acc["surface"].map(surface_group)
acc["lit_cat"] = acc["lit"].map(lit_flag)


print(acc["surface_cat"].value_counts().to_string())
print()
print(acc["lit_cat"].value_counts().to_string())

surface_cat
smooth         34573
unknown         1740
rough_paved     1563
unpaved           16
other              4

lit_cat
lit        37132
unknown      565
unlit        199


In [18]:
import math


def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = successes / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (centre - margin, centre + margin)


known = acc[acc["surface_cat"].isin(["smooth", "rough_paved"])]

print("KSI rate by surface, all crashes:")
for cat in ["rough_paved", "smooth"]:
    g = known[known["surface_cat"] == cat]
    n, k = len(g), int(g["is_ksi"].sum())
    lo, hi = wilson_ci(k, n)
    print(f"  {cat:12s} n={n:6,}  KSI {k/n:6.2%} [{lo:.2%}-{hi:.2%}]")

print("\nKSI rate by surface, solo crashes only (H2):")
solo = known[known["collision_type"] == "Bicycle Only"]
for cat in ["rough_paved", "smooth"]:
    g = solo[solo["surface_cat"] == cat]
    n, k = len(g), int(g["is_ksi"].sum())
    lo, hi = wilson_ci(k, n)
    print(f"  {cat:12s} n={n:6,}  KSI {k/n:6.2%} [{lo:.2%}-{hi:.2%}]")

print("\nSolo share of crashes by surface:")
print((known.groupby("surface_cat")["collision_type"]
       .apply(lambda s: (s == "Bicycle Only").mean() * 100).round(2).to_string()))

KSI rate by surface, all crashes:
  rough_paved  n= 1,563  KSI 10.62% [9.19%-12.25%]
  smooth       n=34,573  KSI 13.15% [12.79%-13.51%]

KSI rate by surface, solo crashes only (H2):
  rough_paved  n=   215  KSI 18.60% [13.97%-24.34%]
  smooth       n= 4,728  KSI 22.29% [21.13%-23.50%]

Solo share of crashes by surface:
surface_cat
rough_paved    13.76
smooth         13.68


### 6.1 H2 — surface hazards — cannot be tested with this source

`surface` is known for 99.5% of crashes after snapping to the re-downloaded graph.
Cobblestone and sett surfaces (`rough_paved`) account for 1,564 crashes.

| | Crashes | KSI rate |
|---|---|---|
| `rough_paved` | 1,564 | **10.61%** [9.18–12.24] |
| `smooth` (asphalt, concrete) | 34,572 | **13.15%** [12.79–13.51] |

Rough surfaces carry a *lower* KSI rate, and the intervals do not overlap. The
same direction holds among solo crashes (18.60% vs 22.29%), though there the
intervals do overlap.

**The direct test is not available.** H2 predicts that rough surfaces cause
single-bicycle falls. In this data solo crashes are 13.75% of crashes on rough
surfaces against 13.68% on smooth — no difference. But that comparison cannot
distinguish "surface does not cause falls" from "falls caused by surface are not
in this dataset."

The Unfallatlas records police-attended crashes. Among cyclist crashes with no
external vehicle involved, roughly 98% never enter the statistics (Juhra et al.,
UKM / Polizei Münster / UDV, 2009–2010; see `01_eda_and_analysis`). Police attend
when liability is at stake, which for a solo fall means almost never. The
mechanism H2 describes therefore operates almost entirely in the unobserved
layer, and the identical solo shares reflect recording selection rather than
surface.

**H2 is untestable with this source, not refuted.** Testing it needs hospital or
ambulance records, which carry the crash mechanism but no coordinates.

**The KSI difference still needs an explanation, and two survive.** Speed —
cobblestone is uncomfortable, riders slow down, and lower impact energy follows;
consistent with the gradient in `04_severity_model`, where `living_street`
carried the lowest KSI rate of any class at 9.81%. Confounding — rough surfaces
in Berlin concentrate in traffic-calmed historic streets, so this compares street
types as much as surfaces.

**The 22.01% KSI rate for solo crashes is no longer puzzling.** A solo fall
reaches the police record only when it is severe enough that someone calls, so
recorded solo crashes are a severity-selected subsample of a much larger
unobserved population. The high rate is a property of the recording process.

Tram rails (`railway=tram`) remain a candidate mechanism but are absent from the
bicycle network graph, and would face the same recording problem.

### 6.2 Lighting cannot be tested, but its absence is informative

Only **198 crashes** occurred on segments tagged `lit=no`, against 37,132 on lit
segments. Intersected with night hours this leaves roughly a dozen cases —
nothing is measurable.

The imbalance is itself the finding. Unlit edges are **7.8% of the network** but
**0.5% of crashes** — under-represented by a factor of 15. That cannot mean unlit
segments are fifteen times safer; it means almost nobody rides them.

This is measured justification for the hard constraint rather than a learned
multiplier: the routing engine currently favours unlit paths because they carry
no recorded crashes, and they carry no recorded crashes because they carry no
cyclists.